### Tools

Models can request to call tools that perform tasks such as fetching data from a database, searching the web, or running code.

Tools are pairings of:
1. A schema, including the name of the tool, a description, and/or argument definitions (often a JSON schema)
2. A function or a co-routine to execute

In [3]:
import os
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
load_dotenv()

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

model = init_chat_model("groq:llama-3.3-70b-versatile")
response = model.invoke("Today's weather in Kolkata!")

response

AIMessage(content="I'm not able to provide real-time weather updates. However, I can suggest some ways for you to find out the current weather in Kolkata.\n\nYou can check the weather forecast on various websites such as:\n\n1. AccuWeather (accuweather.com)\n2. Weather.com (weather.com)\n3. India Meteorological Department (imd.gov.in)\n4. BBC Weather (bbc.com/weather)\n\nYou can also download mobile apps like Dark Sky, Weather Underground, or The Weather Channel to get the current weather conditions and forecast for Kolkata.\n\nPlease note that the weather in Kolkata can be hot and humid during the summer months (March to May) and mild during the winter months (December to February). The city experiences a monsoon season from June to September, with heavy rainfall and high humidity.\n\nIf you need more specific information, please let me know and I'll do my best to help!", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 180, 'prompt_tokens': 41, 'total_tok

In [4]:
from langchain.tools import tool

@tool
def get_weather(location: str) -> str:
    """Get weather at a location"""
    return f"It's sunny in {location}."

model_with_tools = model.bind_tools([get_weather])

In [5]:
response = model_with_tools.invoke("What's the weather in Boston?")

print(response)

# View tool calls made by model
for tool_call in response.tool_calls:
    print(f"Tool: {tool_call["name"]}")
    print(f"Args: {tool_call["args"]}")

content='' additional_kwargs={'tool_calls': [{'id': 'yxyg0mhqq', 'function': {'arguments': '{"location":"Boston"}', 'name': 'get_weather'}, 'type': 'function'}]} response_metadata={'token_usage': {'completion_tokens': 14, 'prompt_tokens': 217, 'total_tokens': 231, 'completion_time': 0.036257952, 'completion_tokens_details': None, 'prompt_time': 0.015734741, 'prompt_tokens_details': None, 'queue_time': 0.058685529, 'total_time': 0.051992693}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_dae98b5ecb', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'} id='lc_run--019fb6ff-885e-7e71-bacd-37a49325169e-0' tool_calls=[{'name': 'get_weather', 'args': {'location': 'Boston'}, 'id': 'yxyg0mhqq', 'type': 'tool_call'}] invalid_tool_calls=[] usage_metadata={'input_tokens': 217, 'output_tokens': 14, 'total_tokens': 231}
Tool: get_weather
Args: {'location': 'Boston'}


#### Tool Execution Loops

In [6]:
# Model aggregates tool calls
messages = [{"role": "user", "content": "What's the weather in Kolkata?"}]
ai_msg = model_with_tools.invoke(messages)
messages.append(ai_msg)

# Tools execution and result collection
for tool_call in ai_msg.tool_calls:
    tool_result = get_weather.invoke(tool_call)
    messages.append(tool_result)

# Pass the result back to model for final response
final_response = model_with_tools.invoke(messages)
print(final_response.text)

I'm glad I could help with the weather in Kolkata. If you have any more questions or need further assistance, feel free to ask!


In [7]:
messages

[{'role': 'user', 'content': "What's the weather in Kolkata?"},
 AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'xmqqmpa2h', 'function': {'arguments': '{"location":"Kolkata"}', 'name': 'get_weather'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 15, 'prompt_tokens': 217, 'total_tokens': 232, 'completion_time': 0.044896607, 'completion_tokens_details': None, 'prompt_time': 0.051087027, 'prompt_tokens_details': None, 'queue_time': 0.050477963, 'total_time': 0.095983634}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_dae98b5ecb', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019fb705-2eee-7492-82aa-1ddadeb61188-0', tool_calls=[{'name': 'get_weather', 'args': {'location': 'Kolkata'}, 'id': 'xmqqmpa2h', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 217, 'output_tokens': 15, 'total_tokens': 232}),
 ToolMessage(content="I